In [ ]:
from sqlalchemy import create_engine
import pandas as pd

username = "root"
password = "1234"
host = "localhost"
port = "3306"
database = "cart2insights"

engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}")

SyntaxError: invalid syntax (2773629956.py, line 3)

In [4]:
#Total Revenue
query = "SELECT SUM(price + freight_value) AS total_revenue FROM order_items;"
result = pd.read_sql(query, con=engine)
print(result)

   total_revenue
0   1.584355e+07


In [5]:
#Total Orders
query = "SELECT COUNT(DISTINCT order_id) AS total_orders FROM orders;"
result = pd.read_sql(query, con=engine)
print(result)

   total_orders
0         99441


In [6]:
#Total Customers
query = "SELECT COUNT(DISTINCT customer_unique_id) AS total_customers FROM customers;"
result = pd.read_sql(query, con=engine)
print(result)

   total_customers
0            96096


In [7]:
#Total Sellers
query = "SELECT COUNT(DISTINCT seller_id) AS total_sellers FROM sellers;"
result = pd.read_sql(query, con=engine)
print(result)

   total_sellers
0           3095


In [8]:
#Average Order Value
query = """ SELECT AVG(order_total) AS avg_order_value
FROM (SELECT order_id, SUM(price + freight_value) AS order_total
FROM order_items GROUP BY order_id) AS order_totals;"""
result = pd.read_sql(query, con=engine)
print(result)


   avg_order_value
0       160.577638


In [9]:
#Average Review Score
query = "SELECT AVG(review_score) AS avg_review_score FROM order_reviews;"
result = pd.read_sql(query, con=engine)
print(result)

   avg_review_score
0            4.0864


In [17]:
query = """
SELECT 
    ct.product_category_name_english AS category,
    SUM(oi.price + oi.freight_value) AS revenue
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
JOIN category_translation ct ON p.product_category_name = ct.product_category_name
GROUP BY ct.product_category_name_english
ORDER BY revenue DESC
LIMIT 10;
"""
category_revenue = pd.read_sql(query, con=engine)
print(category_revenue)


                category     revenue
0          health_beauty  1441248.07
1          watches_gifts  1305541.61
2         bed_bath_table  1241681.72
3         sports_leisure  1156656.48
4  computers_accessories  1059272.40
5        furniture_decor   902511.79
6             housewares   778397.77
7             cool_stuff   719329.95
8                   auto   685384.32
9           garden_tools   584219.21


In [18]:
query = """
SELECT 
    product_id,
    COUNT(*) AS times_ordered,
    SUM(price + freight_value) AS total_revenue
FROM order_items
GROUP BY product_id
ORDER BY times_ordered DESC
LIMIT 10;
"""
top_products = pd.read_sql(query, con=engine)
print(top_products)

                         product_id  times_ordered  total_revenue
0  aca2eb7d00ea1a7b8ebd4e68314663af            527       44820.76
1  99a4788cb24856965c36a24e339b6058            488       51071.60
2  422879e10f46682990de24d770e7f83d            484       34201.26
3  389d119b48cf3043d311335e499d9c6b            392       28682.68
4  368c6c730842d78016ad823897a372db            388       27984.40
5  53759a2ecddad2bb87a079a1f1519f73            373       27268.23
6  d1c427060a0f73f6b889a5c7c61f2ac4            343       60976.03
7  53b36df67ebb7c41585e8d54d6772e08            323       39957.93
8  154e7e31ebfa092203795c972e5804a6            281       10063.11
9  3dd2a17168ec895c781a9191c1e95ad7            274       48212.22


In [19]:
query = """
SELECT 
    c.customer_state,
    SUM(oi.price + oi.freight_value) AS total_revenue,
    COUNT(DISTINCT o.order_id) AS total_orders
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
GROUP BY c.customer_state
ORDER BY total_revenue DESC
LIMIT 10;
"""
sales_by_state = pd.read_sql(query, con=engine)
print(sales_by_state)

  customer_state  total_revenue  total_orders
0             SP     5921678.12         41375
1             RJ     2129681.98         12762
2             MG     1856161.49         11544
3             RS      885826.76          5432
4             PR      800935.44          4998
5             BA      611506.67          3358
6             SC      610213.60          3612
7             DF      353229.44          2125
8             GO      347706.93          2007
9             ES      324801.91          2025


In [ ]:
query = """
SELECT customer_state, COUNT(DISTINCT customer_unique_id) AS customer_count
FROM customers
GROUP BY customer_state
ORDER BY customer_count DESC
LIMIT 10;
"""
customer_dist = pd.read_sql(query, con=engine)
print(customer_dist)

  customer_state  customer_count
0             SP           40302
1             RJ           12384
2             MG           11259
3             RS            5277
4             PR            4882
5             SC            3534
6             BA            3277
7             DF            2075
8             ES            1964
9             GO            1952


In [21]:
query = """
SELECT 
    c.customer_unique_id,
    SUM(oi.price + oi.freight_value) AS total_spent
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
GROUP BY c.customer_unique_id
ORDER BY total_spent DESC
LIMIT 10;
"""
customer_spending = pd.read_sql(query, con=engine)
print(customer_spending)

                 customer_unique_id  total_spent
0  0a0a92112bd4c708ca5fde585afaa872     13664.08
1  da122df9eeddfedc1dc1f5349a1a690c      7571.63
2  763c8b1c9c68a0229c42c9fc6f662b93      7274.88
3  dc4802a71eae9be1dd28f5d788ceb526      6929.31
4  459bef486812aa25204be022145caa62      6922.21
5  ff4159b92c40ebe40454e3e6a7c35ed6      6726.66
6  4007669dec559734d6f53e029e360987      6081.54
7  5d0a2980b292d049061542014e8960bf      4809.44
8  eebb5dda148d3893cdaf5b5ca3040ccb      4764.34
9  48e1ac109decbb87765a3eade6854098      4681.78


In [22]:
query = """
SELECT 
    CASE WHEN order_count > 1 THEN 'Repeat' ELSE 'New' END AS customer_type,
    COUNT(*) AS customer_count
FROM (
    SELECT customer_unique_id, COUNT(DISTINCT customer_id) AS order_count
    FROM customers
    GROUP BY customer_unique_id
) AS customer_orders
GROUP BY customer_type;
"""
repeat_vs_new = pd.read_sql(query, con=engine)
print(repeat_vs_new)

  customer_type  customer_count
0           New           93099
1        Repeat            2997


In [23]:
query = """
SELECT 
    seller_id,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(price + freight_value) AS total_revenue
FROM order_items
GROUP BY seller_id
ORDER BY total_revenue DESC
LIMIT 10;
"""
top_sellers = pd.read_sql(query, con=engine)
print(top_sellers)

                          seller_id  total_orders  total_revenue
0  4869f7a5dfa277a7dca6462dcf3b52b2          1132      249640.70
1  7c67e1448b00f6e969d365cea6b010ab           982      239536.44
2  53243585a1d6dc2643021fd1853d8905           358      235856.68
3  4a3ca9315b744ce9f8e9374361493884          1806      235539.96
4  fa1c13f2614d7b5c4749cbc52fecda94           585      204084.73
5  da8622b14eb17ae2831f4ac5b9dab84a          1314      185192.32
6  7e93a43ef30c4f03f38b393420bc753a           336      182754.05
7  1025f0e2d44d7041d6cf58b6550e0bfa           915      172860.69
8  7a67c85e85bb2ce8582c35f2203ad736          1160      162648.38
9  955fee9216a65b617aa5c0531780ce60          1287      160602.68


In [ ]:
query = """
SELECT 
    product_id,
    COUNT(*) AS times_ordered,
    SUM(price + freight_value) AS total_revenue
FROM order_items
GROUP BY product_id
ORDER BY times_ordered DESC
LIMIT 10;
"""
top_products = pd.read_sql(query, con=engine)
print(top_products)

                         product_id  times_ordered  total_revenue
0  aca2eb7d00ea1a7b8ebd4e68314663af            527       44820.76
1  99a4788cb24856965c36a24e339b6058            488       51071.60
2  422879e10f46682990de24d770e7f83d            484       34201.26
3  389d119b48cf3043d311335e499d9c6b            392       28682.68
4  368c6c730842d78016ad823897a372db            388       27984.40
5  53759a2ecddad2bb87a079a1f1519f73            373       27268.23
6  d1c427060a0f73f6b889a5c7c61f2ac4            343       60976.03
7  53b36df67ebb7c41585e8d54d6772e08            323       39957.93
8  154e7e31ebfa092203795c972e5804a6            281       10063.11
9  3dd2a17168ec895c781a9191c1e95ad7            274       48212.22


In [24]:
query = """
SELECT 
    ct.product_category_name_english AS category,
    COUNT(DISTINCT oi.order_id) AS total_orders,
    SUM(oi.price + oi.freight_value) AS total_revenue
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
JOIN category_translation ct ON p.product_category_name = ct.product_category_name
GROUP BY category
ORDER BY total_revenue DESC
LIMIT 10;
"""
category_performance = pd.read_sql(query, con=engine)
print(category_performance)

                category  total_orders  total_revenue
0          health_beauty          8836     1441248.07
1          watches_gifts          5624     1305541.61
2         bed_bath_table          9417     1241681.72
3         sports_leisure          7720     1156656.48
4  computers_accessories          6689     1059272.40
5        furniture_decor          6449      902511.79
6             housewares          5884      778397.77
7             cool_stuff          3632      719329.95
8                   auto          3897      685384.32
9           garden_tools          3518      584219.21


In [25]:
query = """
SELECT 
    oi.seller_id,
    AVG(r.review_score) AS avg_rating,
    COUNT(DISTINCT oi.order_id) AS total_orders
FROM order_items oi
JOIN order_reviews r ON oi.order_id = r.order_id
GROUP BY oi.seller_id
HAVING total_orders >= 20
ORDER BY avg_rating DESC
LIMIT 10;
"""
seller_ratings = pd.read_sql(query, con=engine)
print(seller_ratings)

                          seller_id  avg_rating  total_orders
0  48efc9d94a9834137efd9ea76b065a38      5.0000            33
1  41c2bad7229b0c25e6becf179ebf63ff      4.9565            20
2  02f5837340d7eb4f653d676c7256523a      4.8333            30
3  d9bd94811c3338dceb4181f3dbc0c73e      4.8197            54
4  d13e50eaa47b4cbe9eb81465865d8cfc      4.8116            67
5  42fa4ee7240e9b8eb4576358ec142ba7      4.8095            21
6  7ade73f1b9b4e965f9009a4c3a7e2c15      4.7778            26
7  3785b653b1b82de85ab47dd139938091      4.7600            21
8  83e197e95a1bbabc8c75e883ed016c47      4.7455            47
9  013900e863eace745d3ec7614cab5b1a      4.7308            23


In [26]:
query = """
SELECT AVG(DATEDIFF(order_delivered_customer_date, order_purchase_timestamp)) AS avg_delivery_days
FROM orders
WHERE order_delivered_customer_date IS NOT NULL;
"""
avg_delivery = pd.read_sql(query, con=engine)
print(avg_delivery)

   avg_delivery_days
0            12.4973


In [27]:
query = """
SELECT 
    CASE WHEN order_delivered_customer_date <= order_estimated_delivery_date THEN 'On-time' ELSE 'Delayed' END AS delivery_status,
    COUNT(*) AS order_count
FROM orders
WHERE order_delivered_customer_date IS NOT NULL
GROUP BY delivery_status;
"""
delivery_status = pd.read_sql(query, con=engine)
print(delivery_status)

  delivery_status  order_count
0         On-time        88649
1         Delayed         7827


In [28]:
query = """
SELECT 
    c.customer_state,
    AVG(DATEDIFF(o.order_delivered_customer_date, o.order_purchase_timestamp)) AS avg_delivery_days
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.order_delivered_customer_date IS NOT NULL
GROUP BY c.customer_state
ORDER BY avg_delivery_days DESC
LIMIT 10;
"""
delivery_by_state = pd.read_sql(query, con=engine)
print(delivery_by_state)

  customer_state  avg_delivery_days
0             RR            29.3415
1             AP            27.1791
2             AM            26.3586
3             AL            24.5013
4             PA            23.7252
5             MA            21.5119
6             SE            21.4627
7             CE            21.2002
8             AC            21.0000
9             PB            20.3888


In [29]:
query = """
SELECT 
    ct.product_category_name_english AS category,
    AVG(r.review_score) AS avg_rating,
    COUNT(*) AS total_reviews
FROM order_reviews r
JOIN order_items oi ON r.order_id = oi.order_id
JOIN products p ON oi.product_id = p.product_id
JOIN category_translation ct ON p.product_category_name = ct.product_category_name
GROUP BY category
HAVING total_reviews >= 50
ORDER BY avg_rating DESC
LIMIT 10;
"""
reviews_by_category = pd.read_sql(query, con=engine)
print(reviews_by_category)

                                category  avg_rating  total_reviews
0                 books_general_interest      4.4463            549
1                costruction_tools_tools      4.4444             99
2                         books_imported      4.4000             60
3                        books_technical      4.3684            266
4                             food_drink      4.3154            279
5                    luggage_accessories      4.3153           1088
6  small_appliances_home_oven_and_coffee      4.3026             76
7                          fashion_shoes      4.2337            261
8                                   food      4.2182            495
9                             cine_photo      4.2055             73
